In [17]:
import tifffile as tiff
import imagecodecs as imagecodecs

import openslide as openslide
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import imageio.v3 as iio
import os
from PIL import Image
import numpy as np
import json

import concurrent.futures

In [ ]:
# --- parameters ---
DATA_PATH = "/Volumes/LabData/BMED6460/beetle-master/data/"
WSI_DIR = DATA_PATH + "images/development/wsis/"
MASK_DIR = DATA_PATH + "annotations/masks/"
JSON_DIR = DATA_PATH + "annotations/jsons/"

# out_dir = DATA_PATH + "images/development/output"
out_dir_imgs = DATA_PATH + "images/development/output/img_patches"
out_dir_masks = DATA_PATH + "images/development/output/mask_patches"
os.makedirs(out_dir_imgs, exist_ok=True)
os.makedirs(out_dir_masks, exist_ok=True)

PATCH_SIZE = 512
STRIDE = PATCH_SIZE   # can set < PATCH_SIZE for overlap
MIN_ANNOT_PIXELS = 0.05  # keep patch if >=5% annotated
BUFFER = 100

In [19]:
# --- Function to Patch a Single WSI ---
def patch_wsi(wsi_path, mask_path, json_path, max_patches=10):
    # --- load WSI ---
    slide = openslide.OpenSlide(wsi_path)
    width, height = slide.dimensions
    wsi_filename = os.path.splitext(os.path.basename(wsi_path))[0]

    # --- load mask ---
    mask = tiff.imread(mask_path)  # shape: (H, W)

    # --- load JSON annotations ---
    with open(json_path, 'r') as f:
        annotations = json.load(f)

    # --- Calculate Bounding Box for All Annotations ---
    x_min, y_min, x_max, y_max = float('inf'), float('inf'), -float('inf'), -float('inf')

    for ann in annotations:
        coords = np.array(ann["coordinates"])
        x_min = min(x_min, np.min(coords[:, 0]))
        y_min = min(y_min, np.min(coords[:, 1]))
        x_max = max(x_max, np.max(coords[:, 0]))
        y_max = max(y_max, np.max(coords[:, 1]))

    # Apply buffer
    x_min = int(max(0, x_min - BUFFER))
    y_min = int(max(0, y_min - BUFFER))
    x_max = int(min(width, x_max + BUFFER))
    y_max = int(min(height, y_max + BUFFER))

    # --- Patch extraction ---
    patch_id = 0
    for y in range(y_min, y_max - PATCH_SIZE + 1, STRIDE):
        for x in range(x_min, x_max - PATCH_SIZE + 1, STRIDE):
            if max_patches is not None and patch_id >= max_patches:
                print(f"Reached max patches for {wsi_filename}. Stopping patching.")
                return

            # --- get mask patch ---
            mask_patch = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

            # skip mostly unannotated patches
            annotated_fraction = np.mean(mask_patch > 0)
            if annotated_fraction < MIN_ANNOT_PIXELS:
                continue

            # --- get WSI patch ---
            wsi_patch = slide.read_region((x, y), 0, (PATCH_SIZE, PATCH_SIZE))
            wsi_patch = wsi_patch.convert("RGB")  # remove alpha

            # --- save ---
            out_img_path = os.path.join(out_dir_imgs, f"{wsi_filename}_{patch_id}.png")
            out_mask_path = os.path.join(out_dir_masks, f"{wsi_filename}_{patch_id}.png")

            # Save image and mask patches
            wsi_patch.save(out_img_path)
            Image.fromarray(mask_patch.astype(np.uint8)).save(out_mask_path)

            patch_id += 1

    print(f"Saved {patch_id} patches for {wsi_filename}.")

In [20]:
# --- Main loop to process all WSIs ---
def process_all_wsis(max_patches_per_wsi=10):
    # List all WSI files in the directory (only .tif files)
    wsi_files = [f for f in os.listdir(WSI_DIR) if f.endswith('.tif')]

    for wsi_file in wsi_files:
        wsi_path = os.path.join(WSI_DIR, wsi_file)
        wsi_filename = os.path.splitext(wsi_file)[0]

        # Corresponding mask and JSON paths (assuming same name)
        mask_path = os.path.join(MASK_DIR, f"{wsi_filename}.tif")
        json_path = os.path.join(JSON_DIR, f"{wsi_filename}.json")

        # Check if corresponding mask and json exist
        if not os.path.exists(mask_path):
            print(f"Warning: Mask file {mask_path} not found, skipping.")
            continue
        if not os.path.exists(json_path):
            print(f"Warning: JSON file {json_path} not found, skipping.")
            continue

        # Process the current WSI
        print(f"Processing {wsi_filename}...")
        patch_wsi(wsi_path, mask_path, json_path, max_patches=max_patches_per_wsi)

    print("Finished processing all WSIs.")

In [21]:
# Run the processing for all WSIs
process_all_wsis()

Processing patient100_wsi1...
Saved 8 patches for patient100_wsi1.
Processing patient101_wsi1...
Saved 4 patches for patient101_wsi1.
Processing patient101_wsi2...
Saved 1 patches for patient101_wsi2.
Processing patient102_wsi1...
Saved 0 patches for patient102_wsi1.
Processing patient103_wsi1...
Reached max patches for patient103_wsi1. Stopping patching.
Processing patient104_wsi1...
Reached max patches for patient104_wsi1. Stopping patching.
Processing patient105_wsi1...
Saved 7 patches for patient105_wsi1.
Processing patient106_wsi1...
Saved 2 patches for patient106_wsi1.
Processing patient106_wsi2...
Saved 5 patches for patient106_wsi2.
Processing patient107_wsi1...
Saved 6 patches for patient107_wsi1.
Processing patient108_wsi1...
Reached max patches for patient108_wsi1. Stopping patching.
Processing patient108_wsi2...
Reached max patches for patient108_wsi2. Stopping patching.
Processing patient109_wsi1...
Reached max patches for patient109_wsi1. Stopping patching.
Processing pat